In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np
import glob
import os
import pdfplumber
import re
import hashlib
import joblib

PDF_FOLDER = "Document_Corpus"
CACHE_DIR = ".search_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_VECTORIZER = os.path.join(CACHE_DIR, "vectorizer.joblib")
CACHE_MATRIX = os.path.join(CACHE_DIR, "tfidf_matrix.joblib")
CACHE_CORPUS = os.path.join(CACHE_DIR, "corpus.joblib")
CACHE_METADATA = os.path.join(CACHE_DIR, "metadata.joblib")
CACHE_HASHES = os.path.join(CACHE_DIR, "file_hashes.joblib")
CACHE_EMBEDDINGS = os.path.join(CACHE_DIR, "embeddings.joblib")

def compute_file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def get_current_hashes(pdf_folder):
    pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
    return {os.path.basename(f): compute_file_hash(f) for f in pdf_files}


def is_cache_valid(current_hashes):
    cache_files = [
        CACHE_VECTORIZER,
        CACHE_MATRIX,
        CACHE_CORPUS,
        CACHE_METADATA,
        CACHE_HASHES,
        CACHE_EMBEDDINGS,
    ]

    if not all(os.path.exists(f) for f in cache_files):
        return False

    old_hashes = joblib.load(CACHE_HASHES)
    return old_hashes == current_hashes


def extract_from_pdfs(pdf_folder):
    corpus = []
    metadata = []
    pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))

    for pdf_path in pdf_files:
        filename = os.path.basename(pdf_path)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages):
                    text = page.extract_text(x_tolerance=2.0)
                    if text and text.strip():
                        corpus.append(text)
                        metadata.append(
                            {
                                "filename": filename,
                                "page": page_num + 1,
                                "raw_text": text.strip(),
                            }
                        )
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    return corpus, metadata

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

current_hashes = get_current_hashes(PDF_FOLDER)
if is_cache_valid(current_hashes):
    print("Cache is valid.")
    vectorizer = joblib.load(CACHE_VECTORIZER)
    Tf_Idfmatrix = joblib.load(CACHE_MATRIX)
    corpus = joblib.load(CACHE_CORPUS)
    metadata = joblib.load(CACHE_METADATA)
    print(f"Loaded {len(corpus)} pages from cache")

else:
    print("Changes detected (or first run). Rebuilding index")
    corpus, metadata = extract_from_pdfs(PDF_FOLDER)

    vectorizer = TfidfVectorizer()
    Tf_Idfmatrix = vectorizer.fit_transform(corpus)
    doc_embeddings = semantic_model.encode(corpus, show_progress_bar=True, normalize_embeddings=True)

    joblib.dump(vectorizer, CACHE_VECTORIZER)
    joblib.dump(Tf_Idfmatrix, CACHE_MATRIX)
    joblib.dump(corpus, CACHE_CORPUS)
    joblib.dump(metadata, CACHE_METADATA)
    joblib.dump(current_hashes, CACHE_HASHES)
    joblib.dump(doc_embeddings, CACHE_EMBEDDINGS)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cache is valid.
Loaded 92 pages from cache


In [ ]:
def kwic(text, query, window_words=10, max_snippets=5):
    text_clean = text.replace("\n", " ")
    words = text_clean.split()

    query_terms = {
        w.lower()
        for w in re.findall(r"\b[a-z0-9]+\b", query.lower())
        if w.lower() not in ENGLISH_STOP_WORDS
    }

    def is_match(word):
        clean_word = re.sub(r"[^a-z0-9]", "", word.lower())
        for term in query_terms:
            if not term:
                continue
            if len(term) <= 3:
                if clean_word == term:
                    return True
            else:
                if term in clean_word:
                    return True
        return False

    match_indices = [idx for idx, word in enumerate(words) if is_match(word)]

    if not match_indices:
        return " ".join(words[:30]) + "..."

    snippets = []
    last_end_idx = -1
    snippets_count = 0

    for match_idx in match_indices:
        if snippets_count >= max_snippets:
            break

        start_idx = max(0, match_idx - window_words)
        end_idx = min(len(words), match_idx + window_words + 1)

        if start_idx < last_end_idx:
            continue
        
        snippet_list = []
        for idx in range(start_idx, end_idx):
            word = words[idx]
            if is_match(word):
                # 💡 Highlight ONLY the matched letters inside the word, preserving original casing
                highlighted_word = word
                # Sort query terms by length descending to replace longer matches first
                for term in sorted(query_terms, key=len, reverse=True):
                    if term and term in highlighted_word.lower():
                        pattern = re.compile(re.escape(term), re.IGNORECASE)
                        highlighted_word = pattern.sub(lambda m: f"**{m.group(0)}**", highlighted_word)
                snippet_list.append(highlighted_word)
            else:
                snippet_list.append(word)
                
        prefix = "... " if start_idx > 0 else ""
        suffix = " ..." if end_idx < len(words) else ""
        snippets.append(prefix + " ".join(snippet_list) + suffix)
        
        last_end_idx = end_idx
        snippets_count += 1

    return "\n ".join(snippets)

In [6]:
def search(query, top_n=5, keyword_weight=0.3, semantic_weight=0.7):
    query_vector = vectorizer.transform([query])
    keyword_scores = cosine_similarity(query_vector, Tf_Idfmatrix).flatten()
    query_embedding = semantic_model.encode([query], normalize_embeddings=True)
    semantic_scores = np.dot(doc_embeddings, query_embedding.T).flatten()

    if keyword_scores.max() > 0:
        keyword_norm = keyword_scores / keyword_scores.max()
    else:
        keyword_norm = keyword_scores

    if semantic_scores.max() > 0:
        semantic_norm = semantic_scores / semantic_scores.max()
    else:
        semantic_norm = semantic_scores

    combined_scores = (keyword_weight * keyword_norm) + (
        semantic_weight * semantic_norm
    )

    top_idx = np.argsort(combined_scores)[::-1][:top_n]

    match_found = False
    for rank, idx in enumerate(top_idx):
        score = combined_scores[idx]
        kw_score = keyword_scores[idx]
        sem_score = semantic_scores[idx]

        if score > 0:
            match_found = True
            meta = metadata[idx]
            print(
                f"Rank {rank + 1} - Combined: {score:.4f}  (Keyword: {kw_score:.4f} | Semantic: {sem_score:.4f})"
            )
            print(f"File: {meta['filename']}(Page {meta['page']})")
            snippet = kwic(meta["raw_text"], query, window_words=10)
            print(f"Snippet: {snippet}")

    if not match_found:
        print("No matching document found with score > 0.")

    return top_idx

In [8]:
results = search("finding unusual patterns in data",keyword_weight=0.0, semantic_weight=1.0)

Rank 1 - Combined: 0.9504  (Keyword: 0.0667 | Semantic: 0.4700)
File: Deep Learning for Anomaly Detection.pdf(Page 3)
Snippet: Deep *Learning* for Anomaly Detection: A Review 38:3 • Unknownness. Anomalies are '
 ' with many unknowns, e.g., *instances* with un- known abrupt behaviors, *data* structures, and distributions. They *remain* unknown until actually occur, such '
 ' may demonstrate completely different abnormal characteristics from another class of *anomalies.Forexample,invideosurveillance,theabnormaleventsrobbery,trafficaccidents* and burglary are visually highly different. • Rarity and class '
 ' not impossible, to collect a large amount of labeled abnormal *instances.* This results *in* the unavailability of large-scale labeled *data* *in* '
 ' anomalies is normally much more costly than that of normal *instances.* • Diverse types of anomaly. Three completely different types of '
Rank 2 - Combined: 0.8743  (Keyword: 0.0799 | Semantic: 0.3856)
File: DeepAnT_A_Deep_Learning_Ap